# Chapter 2 — Probability and Statistics

**Math for ML Engineers** · companion notebook

This notebook mirrors §2.2–§2.4 of the book (MLE, MAP estimation, and conjugate Bayesian updating), the machinery behind every loss function and regularizer you've ever tuned by feel. **Key takeaway: choosing a loss function is choosing a likelihood, and adding a regularizer is choosing a prior — MLE and MAP are not separate from deep learning, they *are* deep learning training, just with different names.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## From likelihood to MAP

Given i.i.d. data $x_1, \dots, x_n$ drawn from $\mathcal{N}(\mu, \sigma^2)$, the **maximum likelihood estimate (MLE)** maximizes $p(x \mid \mu, \sigma^2)$ over $\mu, \sigma^2$. For a Gaussian this has a closed form:

$$\hat\mu = \frac{1}{n}\sum_{i=1}^n x_i, \qquad \hat\sigma^2 = \frac{1}{n}\sum_{i=1}^n (x_i - \hat\mu)^2$$

**MAP estimation** (§2.3) instead maximizes the *posterior* $p(\mu \mid x) \propto p(x \mid \mu)\,p(\mu)$. Placing a Gaussian prior $\mu \sim \mathcal{N}(\mu_0, \tau^2)$ on the mean and combining it with a Gaussian likelihood $x_i \mid \mu \sim \mathcal{N}(\mu, \sigma^2)$ gives a posterior that is *also* Gaussian (conjugacy):

$$\mu \mid x \sim \mathcal{N}(\mu_{\text{post}}, \sigma_{\text{post}}^2), \qquad \sigma_{\text{post}}^2 = \left(\frac{n}{\sigma^2} + \frac{1}{\tau^2}\right)^{-1}, \qquad \mu_{\text{post}} = \sigma_{\text{post}}^2\left(\frac{\sum_i x_i}{\sigma^2} + \frac{\mu_0}{\tau^2}\right)$$

Read $\mu_{\text{post}}$ as a **precision-weighted average** of the data mean and the prior mean $\mu_0$: the term $n/\sigma^2$ is how much the data "votes," and $1/\tau^2$ is how much the prior "votes." As $\tau^2 \to \infty$ (a flat, uninformative prior) MAP collapses onto the MLE. As $\tau^2 \to 0$ (a very confident prior) MAP collapses onto $\mu_0$ regardless of the data. This is exactly Ridge regression in disguise — the shrinkage penalty is the negative log-prior.

In [ ]:
# Synthetic data: true mean is 5.0, but we only observe a small noisy sample
true_mu = 5.0
sigma2 = 4.0          # known observation variance
n = 8
x = np.random.normal(true_mu, np.sqrt(sigma2), size=n)

# Prior belief: mean is near 0 (e.g. a naive "default weight" prior), fairly weak
mu0 = 0.0

print(f"data: {np.round(x, 3)}")
print(f"MLE mean of x: {x.mean():.3f}  (true mu = {true_mu})")

## Implementing MLE and MAP

`gaussian_mle` reproduces the two closed-form estimators above directly. `gaussian_map` implements the precision-weighted posterior mean and variance. We'll then sweep the prior variance $\tau^2$ from very tight (strong prior, heavy shrinkage) to very loose (weak prior, MAP ≈ MLE) and watch $\mu_{\text{post}}$ move along that path.

We also implement the **Beta-Binomial conjugate model** (§2.4): a Beta$(\alpha_0, \beta_0)$ prior over a coin's bias, updated by observed heads/tails into a Beta$(\alpha_0 + \text{heads},\, \beta_0 + \text{tails})$ posterior — the discrete-outcome analogue of the same prior-times-likelihood-equals-posterior idea.

In [ ]:
def gaussian_mle(x):
    """MLE estimates: mu_hat = mean(x), sigma2_hat = mean((x - mu_hat)**2)."""
    mu = x.mean()
    sigma2 = np.mean((x - mu) ** 2)
    return mu, sigma2


def gaussian_map(x, mu0, tau2, sigma2):
    """
    Prior:      mu ~ N(mu0, tau2)
    Likelihood: x_i | mu ~ N(mu, sigma2)
    Posterior:  mu | x ~ N(mu_post, sigma2_post)
    where sigma2_post = 1 / (n/sigma2 + 1/tau2), mu_post = sigma2_post * (sum(x)/sigma2 + mu0/tau2)
    Reduces to Ridge regression: MAP = (precision-weighted) average of prior and data.
    """
    n = len(x)
    prec_post = n / sigma2 + 1 / tau2
    sigma2_post = 1 / prec_post
    mu_post = sigma2_post * (x.sum() / sigma2 + mu0 / tau2)
    return mu_post, sigma2_post


def beta_posterior(heads, tails, alpha0=1.0, beta0=1.0):
    """Beta-Binomial conjugate model: prior Beta(a0,b0) -> posterior Beta(a0+heads, b0+tails)."""
    return alpha0 + heads, beta0 + tails


mle_mu, mle_sigma2 = gaussian_mle(x)
print(f"MLE:  mu_hat = {mle_mu:.3f}, sigma2_hat = {mle_sigma2:.3f}")

tau2_demo = 1.0
map_mu, map_sigma2 = gaussian_map(x, mu0, tau2_demo, sigma2)
print(f"MAP (tau2={tau2_demo}): mu_post = {map_mu:.3f}, sigma2_post = {map_sigma2:.3f}")

## What we're about to plot

**Left:** $\mu_{\text{post}}$ as a function of $\tau^2$ on a log scale, with horizontal reference lines for the MLE (data-only) and the prior mean $\mu_0$. As $\tau^2$ shrinks, the posterior mean is pulled off the MLE and shrunk toward $\mu_0$ — the signature of regularization.

**Right:** the Beta-Binomial prior vs. posterior density after observing a small run of coin flips, showing the same prior→data blending mechanism in the discrete-outcome case.

In [ ]:
from scipy import stats  # only used for the Beta pdf curve

tau2_grid = np.logspace(-2, 3, 200)
mu_posts = np.array([gaussian_map(x, mu0, t2, sigma2)[0] for t2 in tau2_grid])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(tau2_grid, mu_posts, color='C0', lw=2, label=r'MAP $\mu_{post}(\tau^2)$')
ax.axhline(mle_mu, color='C1', ls='--', lw=1.5, label=f'MLE = {mle_mu:.2f}')
ax.axhline(mu0, color='C2', ls=':', lw=1.5, label=f'prior mean $\\mu_0$ = {mu0:.2f}')
ax.set_xscale('log')
ax.set_xlabel(r'prior variance $\tau^2$ (log scale)')
ax.set_ylabel(r'posterior mean $\mu_{post}$')
ax.set_title('MAP shrinkage as prior strength varies')
ax.legend()

# Beta-Binomial: observe a short run of coin flips from a biased coin
true_p = 0.7
flips = np.random.binomial(1, true_p, size=12)
heads, tails = flips.sum(), len(flips) - flips.sum()
a0, b0 = 1.0, 1.0  # uniform prior
a_post, b_post = beta_posterior(heads, tails, a0, b0)

p_grid = np.linspace(0, 1, 300)
ax = axes[1]
ax.plot(p_grid, stats.beta.pdf(p_grid, a0, b0), color='C2', ls=':', lw=2, label=f'prior Beta({a0:.0f},{b0:.0f})')
ax.plot(p_grid, stats.beta.pdf(p_grid, a_post, b_post), color='C0', lw=2,
        label=f'posterior Beta({a_post:.0f},{b_post:.0f})')
ax.axvline(true_p, color='k', ls='--', lw=1, label=f'true p = {true_p}')
ax.set_xlabel('coin bias p')
ax.set_ylabel('density')
ax.set_title(f'Beta-Binomial update after {heads} heads / {tails} tails')
ax.legend()

plt.tight_layout()
plt.show()

print(f"As tau2 -> 0:   mu_post -> {gaussian_map(x, mu0, 1e-6, sigma2)[0]:.4f}  (approaches prior mean {mu0})")
print(f"As tau2 -> inf: mu_post -> {gaussian_map(x, mu0, 1e6, sigma2)[0]:.4f}  (approaches MLE {mle_mu:.4f})")
print(f"Beta posterior mean: {a_post / (a_post + b_post):.3f}  (data-only rate: {heads / len(flips):.3f})")

## Production insight: MAP *is* weight decay

This is not an analogy — it's the same optimization problem. Training a neural network with squared-error loss and L2 weight decay minimizes

$$\mathcal{L}(w) = \underbrace{\sum_i (y_i - f_w(x_i))^2}_{\text{negative log-likelihood}} + \underbrace{\lambda \lVert w \rVert_2^2}_{\text{negative log-prior}}$$

which is *exactly* the MAP objective $-\log p(y \mid w) - \log p(w)$ under a Gaussian likelihood and a zero-mean Gaussian prior $w \sim \mathcal{N}(0, \tau^2 I)$. Matching terms shows the weight-decay coefficient in every PyTorch/TensorFlow optimizer is directly tied to the prior variance:

$$\lambda \;\propto\; \frac{1}{\tau^2}$$

So **a small $\tau^2$ (a tight, confident prior that weights should stay near zero) behaves exactly like a large weight-decay coefficient** — both pull parameters hard toward zero regardless of what the data says. Conversely, `weight_decay=0` is precisely $\tau^2 \to \infty$: a flat, uninformative prior, and training reduces to plain MLE. When you tune `weight_decay` in `torch.optim.AdamW`, you are tuning the width of an implicit Gaussian prior over every weight in the network — the exact $\tau^2$ sweep plotted above, just relabeled.

In [ ]:
# Make the tau2 <-> lambda correspondence numerically explicit.
# MAP objective: sum((x_i - mu)^2) / sigma2  +  (mu - mu0)^2 / tau2
# Weight-decay-style objective: sum((x_i - mu)^2) / sigma2  +  lam * (mu - mu0)^2
# => lam = 1 / tau2. We confirm this by minimizing the weight-decay objective directly
# and checking it lands on the same mu_post that gaussian_map() derives analytically.

def map_via_weight_decay(x, mu0, lam, sigma2, mu_grid):
    """Grid-search the weight-decay-style objective; should match gaussian_map's closed form."""
    obj = np.array([
        np.sum((x - mu) ** 2) / sigma2 + lam * (mu - mu0) ** 2
        for mu in mu_grid
    ])
    return mu_grid[np.argmin(obj)]

mu_grid = np.linspace(-2, 8, 20001)

for tau2_test in [0.01, 0.1, 1.0, 10.0, 1000.0]:
    lam = 1.0 / tau2_test
    mu_closed_form, _ = gaussian_map(x, mu0, tau2_test, sigma2)
    mu_grid_search = map_via_weight_decay(x, mu0, lam, sigma2, mu_grid)
    print(f"tau2={tau2_test:>8.2f}  lambda={lam:>8.3f}  "
          f"MAP closed-form={mu_closed_form:.4f}  weight-decay grid-search={mu_grid_search:.4f}")

## Exercise

1. In `gaussian_map`, fix $\tau^2 = 1$ and re-derive $\mu_{\text{post}}$ symbolically as $n \to \infty$ with $\sigma^2$ held fixed. Show that the prior's influence vanishes at rate $O(1/n)$, and connect this to §2.5's discussion of MAP estimation — why does more data always eventually dominate a fixed prior, no matter how confident (how small $\tau^2$) that prior is? What does this imply about a bug where `weight_decay` is accidentally left nonzero on a *huge* production dataset?

2. Extend `beta_posterior` to compute the posterior *predictive* probability of the next flip being heads, $\mathbb{E}[p \mid x] = \alpha_{\text{post}} / (\alpha_{\text{post}} + \beta_{\text{post}})$, and compare it against the plug-in MLE rate `heads / (heads + tails)` for a very small sample (e.g. 2 flips, both heads). Why does the Bayesian estimate avoid the MLE's degenerate $\hat p = 1.0$ prediction? Use this to write a short pre-launch checklist entry (in the style of Appendix D) for validating priors/regularizers before training on a new, possibly tiny, dataset.